# NanoString: is the interferon score robust to how it is computed?
### Notebook `04_ifn_score_robustness` (Python; runs after `01` and reads the same files as `02`/`03`)

**What this does.** Takes the raw counts of the RGM set (from `nanostring_master.csv`) and of the JDM workbook, and asks one question of each:
*if I had made a different, equally defensible choice, would the interferon score change?* Choices tested: which housekeeping genes, how low counts are floored, whether background is subtracted, and which score formula is used.

**What this does NOT do.** It does not pool RGM and JDM, it does not test anything clinical, and it does not touch the RNA-seq. It only decides whether the primary score used in notebook 03 (`geomean28`) can be trusted before you build on it.

**Ideas borrowed from** nanoR (housekeeping correlation and normalisation checks), nf-core/nanostring (several score methods run on one gene list, PLAGE among them), NACHO/NanoTube (background handling options), and a worked SLE example (252 patients, raw counts, Simoa IFN-alpha) used to test this very code.

## Part 1 · Settings
🎯 **Goal:** every choice lives here. 💡 **Why:** the decision rules (`RHO_OK`, `HK_SCORE_FLAG`) are fixed *before* looking at any result, so the verdicts cannot be tuned afterwards.

In [ ]:
import os, re, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

# ---- where things are -------------------------------------------------------
OUT        = "/mnt/filesystems/nanostring_out"                 # same folder as notebooks 01-03
MASTER_CSV = os.path.join(OUT, "nanostring_master.csv")
JDM_REGEX  = r"^JDM IFN scores OADR.*\.xlsx$"
JDM_SEARCH = ["/mnt/filesystems", "/mnt/session_data", "/home/jovyan"]
JDM_XLSX   = None            # or a full path
RAW_SHEET  = None            # None = the sheet named like "raw" (not "no form", not "STD")
RUN        = ["RGM", "JDM"]  # drop one to skip it

# ---- which RGM rows ---------------------------------------------------------
RGM_RECORD_TYPES    = ["patient_sample", "healthy_control"]
RGM_REQUIRE_QC_PASS = True
BG_COLUMN           = "neg_control_mean"   # per-sample background from notebook 01; "limit_of_detection" is the stricter choice

# ---- genes ------------------------------------------------------------------
HK    = ["ALAS1", "HPRT1", "TBP", "TUBB"]
IFN28 = ["CXCL10","DDX60","EPSTI1","GBP1","HERC5","HERC6","IFI27","IFI44","IFI44L","IFI6","IFIT1","IFIT2",
         "IFIT3","IFIT5","ISG15","LAMP3","LY6E","MX1","OAS1","OAS2","OAS3","OASL","RSAD2","RTP4","SIGLEC1",
         "SOCS1","SPATS2L","USP18"]
IFN25 = [g for g in IFN28 if g not in ("SIGLEC1", "SPATS2L", "EPSTI1")]          # published 25-gene variant
KIM13 = ["EPSTI1","HERC5","IFI27","IFI44","IFI44L","IFI6","IFIT1","IFIT3","ISG15","MX1","OAS1","OAS3","RSAD2"]  # Kim 2020
NO_IFI27 = [g for g in IFN28 if g != "IFI27"]                                    # IFI27 dominates the STD score

# ---- choices and decision rules (fixed in advance) --------------------------
PRIMARY_FLOOR = 0.5                    # what notebook 03 does: counts below 0.5 become 0.5
FLOORS        = [0.1, 0.5, 1, 5]
LOW_COUNT     = 20                     # a raw count below this is "close to nothing" (used when no background is available)
RHO_OK        = 0.95                   # a variant with rank correlation >= this with the primary score changes nothing that matters
HK_SCORE_FLAG = 0.30                   # housekeeping counts that track the score by more than this (either sign) are flagged
HK_MIN_COUNT  = 50                     # same low-housekeeping cut-off notebook 01 uses

## Part 2 · The functions
🎯 **Goal:** one definition of the score, with the choices as arguments. 💡 **How the primary score is built** (identical to notebook 03 and Kim 2018): scale each sample so the geometric mean of its housekeeping genes is 1000, then take the geometric mean of the 28 interferon genes and divide by 10. The `floor` stops a zero count from sending a geometric mean to zero; `bg` (optional) is a per-sample background subtracted first.

In [ ]:
def normalise(cnt, hk, floor=PRIMARY_FLOOR, bg=None):
    """counts (samples x genes) -> counts scaled so each sample's housekeeping geomean = 1000."""
    c = cnt.astype(float)
    if bg is not None:
        c = c.sub(bg.astype(float), axis=0)
    c = c.clip(lower=floor)
    hkg = np.exp(np.log(c[hk]).mean(axis=1))
    return c.div(hkg, axis=0) * 1000

def geomean_score(n, genes):
    return np.exp(np.log(n[genes]).mean(axis=1)) / 10

def rho(a, b):
    """Spearman rank correlation, pairwise-complete, no scipy needed."""
    x = pd.concat([pd.Series(a).reset_index(drop=True), pd.Series(b).reset_index(drop=True)], axis=1).dropna()
    return np.nan if len(x) < 5 else x.rank().corr().iloc[0, 1]

def plage(n, genes):
    """first principal component of the z-scored log2 gene set (nf-core's PLAGE), sign set so higher = more interferon."""
    X = np.log2(n[genes]); Z = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
    Z = Z.fillna(0).to_numpy()
    U, S, _ = np.linalg.svd(Z, full_matrices=False)
    pc1 = pd.Series(U[:, 0] * S[0], index=n.index)
    return pc1 if rho(pc1, geomean_score(n, genes)) >= 0 else -pc1

def all_variants(cnt, hk, bg=None):
    n = normalise(cnt, hk, PRIMARY_FLOOR, bg)
    v = {
        "geomean28 (primary)":      geomean_score(n, IFN28),
        "geomean25":                geomean_score(n, IFN25),
        "geomean13 (Kim 2020)":     geomean_score(n, KIM13),
        "geomean28 minus IFI27":    geomean_score(n, NO_IFI27),
        "median28 / 10":            n[IFN28].median(axis=1) / 10,
        "mean log2 28":             np.log2(n[IFN28]).mean(axis=1),
        "PLAGE (PC1) 28":           plage(n, IFN28),
    }
    return pd.DataFrame(v)

## Part 3 · Load the two sets
🎯 **Goal:** for each set, a table of raw counts (one row per sample) and, where it exists, a per-sample background.
💡 **Why separate loaders:** the RGM master (from `.RCC` files) carries run-level background and QC; the JDM workbook carries only counts, so background checks are skipped for it, and the notebook says so.

In [ ]:
def find_one(pat):
    hits = []
    for d in JDM_SEARCH:
        if os.path.isdir(d):
            for root, _, files in os.walk(d):
                hits += [os.path.join(root, f) for f in files if re.search(pat, f, re.I) and not f.startswith("~$")]
    if not hits: raise FileNotFoundError(f"no file matching {pat} under {JDM_SEARCH}")
    return sorted(hits)[0]

def load_rgm():
    m = pd.read_csv(MASTER_CSV, low_memory=False)
    keep = pd.Series(True, index=m.index)
    if "record_type" in m: keep &= m["record_type"].isin(RGM_RECORD_TYPES)
    if RGM_REQUIRE_QC_PASS and "qc_pass" in m: keep &= m["qc_pass"].astype(str).str.lower().eq("true")
    if "is_non_blood" in m: keep &= ~m["is_non_blood"].astype(str).str.lower().eq("true")
    genes = IFN28 + HK
    missing = [g for g in genes if g not in m]
    if missing: raise KeyError(f"genes missing from the master table: {missing}")
    m = m[keep].copy()
    cnt = m[genes].apply(pd.to_numeric, errors="coerce")
    ok = cnt.notna().all(axis=1)
    m, cnt = m[ok], cnt[ok]
    bg = pd.to_numeric(m[BG_COLUMN], errors="coerce") if BG_COLUMN in m else None
    if bg is not None and bg.isna().all(): bg = None
    key_cols = [c for c in ["raw_filename", "patient_id", "record_type", "diagnosis1", "specimen_type"] if c in m]
    return cnt.reset_index(drop=True), (None if bg is None else bg.reset_index(drop=True)), m[key_cols].reset_index(drop=True)

def load_jdm():
    path = JDM_XLSX or find_one(JDM_REGEX)
    sheets = pd.ExcelFile(path).sheet_names
    sh = RAW_SHEET or next(s for s in sheets if re.search("raw", s, re.I) and not re.search("no|non|form|std", s, re.I))
    raw = pd.read_excel(path, sheet_name=sh, header=None, dtype=str)
    raw = raw.dropna(how="all")
    genes = set(IFN28 + HK)
    head = raw.head(15)
    hits = [sum(str(x).strip() in genes for x in head.iloc[i]) for i in range(len(head))]
    h = int(np.argmax(hits))
    hdr = [str(x).strip() if pd.notna(x) else f"unnamed_{j}" for j, x in enumerate(head.iloc[h])]
    df = raw.iloc[h + 1:].copy(); df.columns = hdr
    missing = [g for g in IFN28 + HK if g not in df]
    if missing: raise KeyError(f"genes missing from the sheet: {missing}")
    for g in IFN28 + HK: df[g] = pd.to_numeric(df[g], errors="coerce")
    df = df[df[IFN28 + HK].notna().all(axis=1)]
    key_cols = [c for c in ["study sample", "Lab initials", "sample date"] if c in df]
    own = {c: pd.to_numeric(df[c], errors="coerce") for c in ["Geomean 28", "Geomean 25"] if c in df}
    print(f"JDM workbook: {os.path.basename(path)} | sheet: {sh} | header row {h} | {len(df)} samples")
    return df[IFN28 + HK].reset_index(drop=True), None, df[key_cols].reset_index(drop=True), pd.DataFrame(own).reset_index(drop=True)

DATA = {}
if "RGM" in RUN:
    try:
        c, b, k = load_rgm(); DATA["RGM"] = dict(cnt=c, bg=b, key=k, own=None)
        print(f"RGM: {len(c)} samples | background column '{BG_COLUMN}': {'found' if b is not None else 'NOT found, background checks skipped'}")
        if "record_type" in k: print(k["record_type"].value_counts().to_string())
    except Exception as e:
        print("RGM skipped:", type(e).__name__, e)
if "JDM" in RUN:
    try:
        c, b, k, own = load_jdm(); DATA["JDM"] = dict(cnt=c, bg=b, key=k, own=own)
    except Exception as e:
        print("JDM skipped:", type(e).__name__, e)
print("\nsets loaded:", list(DATA))

## Part 4 · The checks
Each set gets four checks. Read them in this order; the verdict row at the end of each part uses the rules from Part 1.

**4a Housekeeping genes.** The score divides every gene by the sample's housekeeping geometric mean, so if those genes drift with interferon, they distort the score. Three things are shown: how variable each housekeeping gene is, whether the four agree with each other, and whether any of them tracks the score. *A housekeeping gene that correlates with the score is not automatically wrong (low-input samples lower everything), but you should know it.* Dropping one housekeeping gene at a time shows how much the score depends on any single one.

**4b Floors and background.** A geometric mean is sensitive to near-zero counts. The floor (0.5 in notebook 03) is varied, and where the RGM files give a per-sample background it is subtracted first.

**4c Score definition.** The same 28 genes scored six other ways. If they all rank the samples the same way, the choice of formula does not matter and the primary score is safe to use.

**4d Which genes sit near the floor.** A gene that is often close to nothing contributes noise to a geometric mean.

In [ ]:
RES = {}

def check_set(name, d):
    cnt, bg = d["cnt"], d["bg"]
    out = {}
    n0 = normalise(cnt, HK, PRIMARY_FLOOR, bg=None)
    primary = geomean_score(n0, IFN28)
    print("="*78); print(f"{name}: {len(cnt)} samples | primary score median {primary.median():.0f}, range {primary.min():.0f} to {primary.max():.0f}")

    # ---- 4a housekeeping
    rows = []
    for g in HK:
        rows.append(dict(gene=g, median_count=cnt[g].median(), CV=cnt[g].std() / cnt[g].mean(),
                         pct_below_min=100 * (cnt[g] < HK_MIN_COUNT).mean(), rho_with_score=rho(cnt[g], primary)))
    hk_tab = pd.DataFrame(rows).round(2)
    hkg = np.exp(np.log(cnt[HK].clip(lower=PRIMARY_FLOOR)).mean(axis=1))
    pair = cnt[HK].rank().corr().to_numpy(); pair_min = pair[np.triu_indices(len(HK), 1)].min()
    loo = {f"drop {g}": rho(geomean_score(normalise(cnt, [h for h in HK if h != g]), IFN28), primary) for g in HK}
    hk_flag = bool((hk_tab["rho_with_score"].abs() > HK_SCORE_FLAG).any())
    print("\n4a housekeeping genes"); print(hk_tab.to_string(index=False))
    print(f"   lowest pairwise rho among the four: {pair_min:.2f} | housekeeping geomean vs score: rho {rho(hkg, primary):.2f}")
    print("   score after dropping one housekeeping gene (rho vs primary):", {k: round(float(v), 3) for k, v in loo.items()})
    out["hk"] = hk_tab

    # ---- 4b floors and background
    frows = []
    for fl in FLOORS:
        s = geomean_score(normalise(cnt, HK, fl), IFN28); r = s / primary
        frows.append(dict(variant=f"floor {fl}", rho_vs_primary=rho(s, primary), median_ratio=r.median(), worst_ratio=r.max() if r.max() - 1 > 1 - r.min() else r.min()))
    if bg is not None:
        for fl in (PRIMARY_FLOOR, 1):
            s = geomean_score(normalise(cnt, HK, fl, bg=bg), IFN28); r = s / primary
            frows.append(dict(variant=f"subtract {BG_COLUMN}, floor {fl}", rho_vs_primary=rho(s, primary), median_ratio=r.median(),
                              worst_ratio=r.max() if r.max() - 1 > 1 - r.min() else r.min()))
    else:
        print("\n   (no per-sample background for this set: background subtraction not tested)")
    fl_tab = pd.DataFrame(frows).round(3)
    print("\n4b floors and background"); print(fl_tab.to_string(index=False))
    out["floor"] = fl_tab

    # ---- 4c score definitions
    V = all_variants(cnt, HK, bg=None)
    vt = pd.DataFrame({"variant": V.columns, "rho_vs_primary": [rho(V[c], V.iloc[:, 0]) for c in V.columns]}).round(3)
    print("\n4c score definitions (all on the same samples)"); print(vt.to_string(index=False))
    out["variants"] = vt
    own = d.get("own")
    if own is not None and len(own.columns):
        for c in own.columns:
            print(f"   workbook's own '{c}' vs primary: rho = {rho(own[c], primary):.3f}")

    # ---- 4d genes near the floor
    thr = (2 * bg) if bg is not None else None
    low = pd.Series({g: ((cnt[g] < 2 * bg) if bg is not None else (cnt[g] < LOW_COUNT)).mean() for g in IFN28}).sort_values(ascending=False)
    what = "below 2x the sample's background" if bg is not None else f"below {LOW_COUNT} raw counts"
    print(f"\n4d share of samples with the gene {what} (top 6):", low.head(6).round(2).to_dict())
    print(f"   overall share of the 28-gene count table {what}: {(low.mean()):.3f}")
    out["low"] = low.round(3)

    # ---- verdict
    worst_floor = fl_tab["rho_vs_primary"].min(); worst_var = vt.loc[vt["variant"] != "geomean28 (primary)", "rho_vs_primary"].min()
    worst_loo = min(loo.values())
    verdict = pd.DataFrame([
        dict(set=name, check="housekeeping: drop one gene", worst_rho=worst_loo, verdict="OK" if worst_loo >= RHO_OK else "LOOK"),
        dict(set=name, check="housekeeping: tracks the score", worst_rho=hk_tab["rho_with_score"].abs().max(), verdict="OK" if not hk_flag else "FLAG"),
        dict(set=name, check="floor / background", worst_rho=worst_floor, verdict="OK" if worst_floor >= RHO_OK else "LOOK"),
        dict(set=name, check="score definition", worst_rho=worst_var, verdict="OK" if worst_var >= RHO_OK else "LOOK"),
    ]).round(3)
    out["verdict"] = verdict; out["scores"] = V; out["primary"] = primary; out["hkg"] = hkg
    print("\nverdicts"); print(verdict.to_string(index=False))
    return out

for nm, d in DATA.items():
    RES[nm] = check_set(nm, d)

### Pictures
Left: each housekeeping gene against the primary score (a flat cloud is good; a slope is the thing to know about). Right: the primary score against each alternative formula (a tight diagonal means the same ranking).

In [ ]:
for nm, r in RES.items():
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    for g in HK:
        ax[0].scatter(DATA[nm]["cnt"][g], r["primary"], s=8, alpha=.5, label=g)
    ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("raw housekeeping count"); ax[0].set_ylabel("primary score")
    ax[0].set_title(f"{nm}: housekeeping genes vs score"); ax[0].legend(markerscale=2)
    V = r["scores"]
    for c in V.columns[1:]:
        ax[1].scatter(V.iloc[:, 0].rank(), V[c].rank(), s=6, alpha=.4, label=c)
    ax[1].plot([0, len(V)], [0, len(V)], color="k", lw=.6)
    ax[1].set_xlabel("rank of primary score"); ax[1].set_ylabel("rank of alternative"); ax[1].set_title(f"{nm}: rank agreement of score definitions")
    ax[1].legend(markerscale=2, fontsize=7)
    plt.tight_layout(); plt.show()

## Part 5 · Save
Three files in `OUT`. The per-sample file carries sample keys (patient IDs or study codes), so keep it on Lifebit. `ifn_robustness_verdicts.csv` has no identifiers.

In [ ]:
os.makedirs(OUT, exist_ok=True)
verd = pd.concat([r["verdict"] for r in RES.values()], ignore_index=True) if RES else pd.DataFrame()
verd.to_csv(os.path.join(OUT, "ifn_robustness_verdicts.csv"), index=False)
for nm, r in RES.items():
    per = pd.concat([DATA[nm]["key"], r["scores"].reset_index(drop=True)], axis=1)
    per.to_csv(os.path.join(OUT, f"ifn_score_variants_{nm}.csv"), index=False)
    pd.concat([r["hk"].assign(table="housekeeping"), r["floor"].assign(table="floor_background"), r["variants"].assign(table="score_definitions")],
              ignore_index=True).to_csv(os.path.join(OUT, f"ifn_robustness_tables_{nm}.csv"), index=False)
print(verd.to_string(index=False))
print("\nwritten to", OUT)

## How to read the verdicts, and what to do next

* **All `OK`:** the primary score does not depend on these choices. Keep `geomean28` as the score in notebooks 03 and 05 and say so in one sentence in the methods.
* **`LOOK` on a floor/background row:** a few samples sit near the detection limit. Rerun notebook 03 with that floor and see whether the clinical correlations move; report both if they do.
* **`LOOK` on a score-definition row:** find the variant in the table. `geomean28 minus IFI27` is the usual culprit, since IFI27 is very high in some patients; if it is the one that disagrees, that is a finding about IFI27, and the geomean is the safer cross-set score (Kim 2018).
* **`FLAG` on housekeeping:** housekeeping counts rise or fall with the score. Look at the left picture. If the housekeeping total also falls in IFN-high samples, divide-by-housekeeping inflates the score a little; the dropped-one-gene and floor rows show whether the ranking survives.
* **Nothing here compares RGM with JDM.** Two sets with different codesets and runs can only be compared through fold changes against each set's own reference, or by reprocessing both from the raw `.RCC` files.